# MRL Eye CNN 재학습 파이프라인 (Colab 버전)

`train_eye_mrl.py` / `mrl_dataset.py` / `eye_preprocess.py` / `mrl_split.py` 4개 스크립트를
실행 순서대로 하나의 노트북에 통합한 버전이다.

**원본 대비 변경한 부분 (그 외 로직은 전부 동일)**
- `sys.path` 조작 및 모듈 간 `import` 제거 — 노트북 한 세션에 전부 정의되므로 불필요
- `config.py`의 `PROJECT_ROOT`(파일 위치 기반 자동 탐색) 대신, Colab에는 그 개념이 없으므로
  경로 셀에서 `MRL_ROOT` / `MANIFEST_DIR` / `ARTIFACT_DIR`을 직접 지정한다
- `mrl_split.py`의 `argparse` 진입점을 `run_manifest_split(mrl_root, out_dir, seed)` 함수로 변경
- `mrl_dataset.py`의 `resolve_mrl_root()`는 위 경로 직접 지정과 역할이 겹쳐 제거

**실행 전 확인할 것**
1. 아래 '경로 설정' 셀의 `MRL_ROOT`를 본인 Drive의 실제 데이터 위치로 수정
2. 런타임 유형을 GPU로 설정 (런타임 → 런타임 유형 변경 → GPU)
3. 임계값과 마찬가지로 `SPLIT_MODE / GRAYSCALE / DO_SHARPEN / INPUT_SIZE`는 실험 대상이며,
   이 노트북은 값을 임의로 확정하지 않는다 — 학습·평가 셀 상단 CONFIG에서 직접 바꿔서 실행한다

## 0. 환경 확인 및 패키지 설치

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Colab 환경:", IN_COLAB)

In [ ]:
!pip install -q opencv-python-headless scikit-learn

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU 사용 가능:", tf.config.list_physical_devices('GPU'))

## 1. 경로 설정

`config.py` 원칙(확정 안 된 값은 넣지 않는다)은 유지하되, 파일 위치 기반 자동 탐색이
Colab에서는 동작하지 않으므로 여기서 직접 지정한다. **이 셀의 `MRL_ROOT`만 본인 환경에 맞게 수정.**

In [ ]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # TODO: 본인 Drive의 실제 데이터 경로로 수정
    MRL_ROOT     = "/content/drive/MyDrive/driver-drowsiness-detection/data/MRL Eye/data"
    MANIFEST_DIR = "/content/drive/MyDrive/driver-drowsiness-detection/manifests"
    ARTIFACT_DIR = "/content/drive/MyDrive/driver-drowsiness-detection/artifacts"
else:
    MRL_ROOT     = "data/MRL Eye/data"
    MANIFEST_DIR = "manifests"
    ARTIFACT_DIR = "artifacts"

os.makedirs(MANIFEST_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

print("MRL_ROOT     :", MRL_ROOT)
print("MANIFEST_DIR :", MANIFEST_DIR)
print("ARTIFACT_DIR :", ARTIFACT_DIR)

## 2. `eye_preprocess` — 학습·추론이 공유하는 전처리

MRL 학습(그레이스케일 PNG)과 추론(YuNet BGR crop)이 서로 다른 전처리를 거치면 도메인 갭이
생긴다. 두 경로가 이 함수 하나를 통과하게 하는 것이 핵심이다.
추론 순서(resize → sharpen)를 그대로 맞췄고, 출력은 0~255 float32 (모델 첫 레이어
`Rescaling(1/255)`이 정규화하므로 여기서는 스케일링하지 않는다).

In [ ]:
import cv2
import numpy as np

# 원본 레포와 동일한 sharpen 커널 (detector.py / 02_INFER 와 일치)
SHARPEN_KERNEL = np.array(
    [[0, -1, 0],
     [-1, 5, -1],
     [0, -1, 0]],
    dtype=np.float32,
)


def sharpen(img: np.ndarray) -> np.ndarray:
    """원본 레포와 동일한 3x3 sharpen."""
    return cv2.filter2D(src=img, ddepth=-1, kernel=SHARPEN_KERNEL)


def preprocess_eye(
    img: np.ndarray,
    size: int = 128,
    grayscale: bool = True,
    do_sharpen: bool = False,
    input_is_bgr: bool = True,
) -> np.ndarray:
    """눈 이미지 한 장을 모델 입력 텐서로 변환한다.

    img              : (H,W) 또는 (H,W,1) 그레이스케일(MRL), 혹은 (H,W,3) 컬러(추론 crop)
    size             : 정사각 리사이즈 한 변 (실험 대상: 96 / 128 / 256)
    grayscale        : True면 1채널, False면 3채널 RGB 출력 (실험 대상)
    do_sharpen       : sharpen 적용 여부. 학습·추론에서 반드시 동일해야 한다 (실험 대상)
    input_is_bgr     : 3채널 입력이 BGR(OpenCV)인지 여부. cv2.imread / YuNet crop은 True

    반환: (size, size, C) float32, 값 범위 0~255
    """
    a = np.asarray(img)

    # 1) 색 공간을 BGR 3채널 작업본으로 통일 (색 연산을 한 곳에서만 하기 위함)
    if a.ndim == 2:
        a = a[:, :, None]
    if a.shape[2] == 1:
        work = cv2.cvtColor(a[:, :, 0], cv2.COLOR_GRAY2BGR)
    elif a.shape[2] == 3:
        work = a if input_is_bgr else cv2.cvtColor(a, cv2.COLOR_RGB2BGR)
    else:
        raise ValueError(f"지원하지 않는 채널 수: {a.shape}")

    if work.dtype != np.uint8:
        work = np.clip(work, 0, 255).astype(np.uint8)

    # 2) 추론(02_INFER)과 동일한 순서: resize -> sharpen
    work = cv2.resize(work, (size, size), interpolation=cv2.INTER_AREA)
    if do_sharpen:
        work = sharpen(work)

    # 3) 출력 채널 선택
    if grayscale:
        g = cv2.cvtColor(work, cv2.COLOR_BGR2GRAY)
        out = g[:, :, None]
    else:
        out = cv2.cvtColor(work, cv2.COLOR_BGR2RGB)

    return out.astype(np.float32)


def channels(grayscale: bool) -> int:
    """모델 Input 채널 수 헬퍼."""
    return 1 if grayscale else 3

## 3. `mrl_split` — subject 독립 manifest 생성

Kaggle 포크(MRL Eye)는 이미지 단위 random split이라 같은 사람 눈이 train/val/test에
동시에 들어가 있다(실측 확인). 사람이 섞이면 test 성능이 과대평가되므로,
파일명의 subject ID로 **사람 단위 분할**을 만든다.

이미지를 물리적으로 복사하지 않고 manifest CSV만 생성한다. 두 버전을 함께 만든다:
- `mrl_manifest_subject.csv` : subject 독립 (본 실험용, 권장)
- `mrl_manifest_random.csv`  : 이미지 단위 random (leakage 영향 비교용)

파일명 규칙(실측 확인): `s0001_01842_0_0_1_0_0_01.png`
= subject_imageid_gender_glasses_eyestate_reflect_light_sensor
(eyestate: 0=Closed, 1=Open / class_idx: Closed=0, Open=1)

In [ ]:
import csv
import glob
import random
from collections import Counter, defaultdict

CLASS_IDX = {"Closed": 0, "Open": 1}
SPLITS = ("train", "val", "test")
TARGET = {"train": 0.70, "val": 0.15, "test": 0.15}


def parse_name(path: str, rel_path: str = None):
    """파일명에서 메타데이터를 파싱. 규칙에 안 맞으면 None.

    path     : 실제 파일 경로(파싱용)
    rel_path : manifest에 저장할 경로. None이면 path 그대로.
               이식성을 위해 MRL 루트 기준 상대경로를 넣는다.
    """
    fn = os.path.basename(path)
    stem = fn[:-4] if fn.lower().endswith(".png") else fn
    parts = stem.split("_")
    if len(parts) != 8:
        return None
    subject, imgid, gender, glasses, eyestate, reflect, light, sensor = parts
    label = "Closed" if eyestate == "0" else "Open"
    return {
        "path": rel_path if rel_path is not None else path,
        "subject": subject,
        "label": label,
        "class_idx": CLASS_IDX[label],
        "eyestate": eyestate,
        "gender": gender,
        "glasses": glasses,
        "reflect": reflect,
        "light": light,
        "sensor": sensor,
    }


def collect(mrl_root: str):
    """MRL 루트(.../MRL Eye/data) 아래 모든 png를 파싱해 레코드 리스트로."""
    rows = []
    odd = 0
    for p in glob.glob(os.path.join(mrl_root, "**", "*.png"), recursive=True):
        rel = os.path.relpath(p, mrl_root).replace("\\", "/")  # OS 무관 상대경로
        r = parse_name(p, rel_path=rel)
        if r is None:
            odd += 1
            continue
        rows.append(r)
    if odd:
        print(f"[경고] 파일명 규칙 불일치 {odd}건은 제외했습니다.")
    if not rows:
        raise SystemExit(f"png를 찾지 못했습니다: {mrl_root}")
    return rows


def subject_split(rows: list) -> dict:
    """subject 단위 결정적 stratified 배정 (RNG 없음, 재현 100%).

    단순 크기 배정은 subject 편중 때문에 glasses/closed 비율이 split마다 틀어진다
    (예: 안경 비율 train 31% vs test 19%). 이를 막기 위해 각 subject를
    (glasses x eyestate) 4개 층으로 분해하고, 4개 층 이미지 수를 split 목표 쿼터에
    맞춰 채운다. 사람은 여전히 한 split에만 들어간다(leakage 0).

    층(strata): 0=안경&Closed, 1=안경&Open, 2=비안경&Closed, 3=비안경&Open
    배정: 이미지 수 내림차순으로, 어떤 층도 목표 쿼터를 넘지 않게(=4개 층 중
          최대 채움비가 가장 작은) split을 고른다. 크기·안경·라벨이 동시에 균형.
    """
    strat = {}
    per_subj = Counter()
    for r in rows:
        su = r["subject"]
        per_subj[su] += 1
        d = strat.setdefault(su, [0, 0, 0, 0])
        glass = (r["glasses"] == "1")
        closed = (r["eyestate"] == "0")
        idx = (0 if glass else 2) + (0 if closed else 1)
        d[idx] += 1

    stot = [sum(strat[su][k] for su in strat) for k in range(4)]
    quota = {s: [stot[k] * TARGET[s] for k in range(4)] for s in SPLITS}

    order = sorted(per_subj, key=lambda s: (-per_subj[s], s))  # 완전 결정적
    cur = {s: [0, 0, 0, 0] for s in SPLITS}
    assign = {}
    for su in order:
        d = strat[su]

        def fill(s):
            return max((cur[s][k] + d[k]) / quota[s][k] if quota[s][k] > 0 else 0.0
                       for k in range(4))

        best = min(SPLITS, key=fill)
        assign[su] = best
        for k in range(4):
            cur[best][k] += d[k]
    return assign


def random_split(rows: list, seed: int = 42):
    """이미지 단위 random split (leakage 재현용). 반환: rows와 같은 순서의 split 라벨."""
    idx = list(range(len(rows)))
    random.Random(seed).shuffle(idx)
    n = len(rows)
    n_tr = int(n * TARGET["train"])
    n_va = int(n * TARGET["val"])
    split_of = [""] * n
    for rank, i in enumerate(idx):
        if rank < n_tr:
            split_of[i] = "train"
        elif rank < n_tr + n_va:
            split_of[i] = "val"
        else:
            split_of[i] = "test"
    return split_of


def report(rows: list, split_key="split"):
    """split별 subject 수 / 이미지 수 / Closed·glasses 비율 + leakage 점검."""
    by_split_subj = defaultdict(set)
    cnt = Counter()
    closed = Counter()
    glass = Counter()
    for r in rows:
        s = r[split_key]
        by_split_subj[s].add(r["subject"])
        cnt[s] += 1
        if r["label"] == "Closed":
            closed[s] += 1
        if r["glasses"] == "1":
            glass[s] += 1
    all_subj = set().union(*by_split_subj.values()) if by_split_subj else set()
    leaked = [su for su in all_subj
              if sum(su in by_split_subj[s] for s in SPLITS) > 1]
    print(f"  [{split_key}]")
    for s in SPLITS:
        n = cnt[s]
        if n == 0:
            continue
        print(f"    {s:5s}: {len(by_split_subj[s]):2d} subj | {n:6d} img "
              f"({n/len(rows)*100:4.1f}%) | Closed {closed[s]/n*100:4.1f}% | "
              f"glasses {glass[s]/n*100:4.1f}%")
    print(f"    >1 split에 걸친 subject(leakage): {len(leaked)} / {len(all_subj)}")


def write_csv(rows: list, out_path: str, split_key="split"):
    cols = ["path", "subject", "label", "class_idx", "eyestate",
            "gender", "glasses", "reflect", "light", "sensor", "split"]
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in rows:
            row = {k: r.get(k, "") for k in cols}
            row["split"] = r[split_key]
            w.writerow(row)
    print("  저장:", out_path, f"({len(rows)} rows)")


def run_manifest_split(mrl_root: str, out_dir: str, seed: int = 42):
    """원본 mrl_split.py의 main()을 argparse 없이 호출 가능하게 만든 버전.
    로직은 동일하며 인자만 함수 파라미터로 받는다.
    """
    rows = collect(mrl_root)
    print(f"총 이미지: {len(rows)}")
    print(f"subject 수: {len(set(r['subject'] for r in rows))}")

    assign = subject_split(rows)
    for r in rows:
        r["split_subject"] = assign[r["subject"]]
    rand = random_split(rows, seed=seed)
    for r, s in zip(rows, rand):
        r["split_random"] = s

    print("\n== subject-independent split ==")
    report(rows, "split_subject")
    print("\n== random split (비교용) ==")
    report(rows, "split_random")

    os.makedirs(out_dir, exist_ok=True)
    print("\n== 저장 ==")
    write_csv(rows, os.path.join(out_dir, "mrl_manifest_subject.csv"), "split_subject")
    write_csv(rows, os.path.join(out_dir, "mrl_manifest_random.csv"), "split_random")

    print("\n== subject 배정 ==")
    for s in SPLITS:
        subs = sorted(su for su, sp in assign.items() if sp == s)
        print(f"  {s}: {' '.join(subs)}")

    return assign

manifest 생성 실행 (한 번만 돌리면 됨 — 이후 셀들은 생성된 CSV를 재사용).

In [ ]:
_ = run_manifest_split(MRL_ROOT, MANIFEST_DIR, seed=42)

## 4. `mrl_dataset` — manifest → `tf.data.Dataset`

이미지 로드 후 위 `preprocess_eye()`를 `tf.py_function`으로 적용한다.
학습과 추론이 완전히 같은 numpy 전처리를 통과하게 만들어 parity를 보장한다
(tf 내장 resize를 따로 쓰면 추론(cv2)과 미묘하게 달라질 수 있어 일부러 cv2 경유).

라벨: class_idx(Closed=0, Open=1) → one-hot 2. 모델 `Dense(2, softmax)`의 class0이
Closed 확률이 되어 기존 추론/EMA/PERCLOS와 그대로 연결된다.

In [ ]:
def _read_manifest(csv_path: str, split: str, mrl_root: str = ""):
    """manifest의 상대경로를 mrl_root와 합쳐 실제 경로로 만든다."""
    paths, labels = [], []
    with open(csv_path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["split"] != split:
                continue
            p = row["path"]
            paths.append(os.path.join(mrl_root, p) if mrl_root else p)
            labels.append(int(row["class_idx"]))
    return paths, labels


def make_dataset(
    csv_path: str,
    split: str,
    size: int = 128,
    grayscale: bool = True,
    do_sharpen: bool = False,
    batch_size: int = 32,
    shuffle: bool = None,
    seed: int = 42,
    mrl_root: str = "",
):
    """manifest의 특정 split을 tf.data로. shuffle 기본값은 train일 때만 True.

    mrl_root : manifest의 상대경로를 붙일 MRL 루트(.../MRL Eye/data).
    """
    if shuffle is None:
        shuffle = (split == "train")

    paths, labels = _read_manifest(csv_path, split, mrl_root)
    if not paths:
        raise ValueError(f"'{split}' split에 해당하는 행이 없습니다: {csv_path}")
    c = channels(grayscale)

    def _load(path, label):
        def _py(p):
            p = p.numpy().decode("utf-8")
            # 디스크의 MRL은 그레이스케일. cv2로 읽어 preprocess 공유.
            img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
            arr = preprocess_eye(img, size=size, grayscale=grayscale,
                                 do_sharpen=do_sharpen, input_is_bgr=True)
            return arr.astype(np.float32)

        img = tf.py_function(_py, [path], tf.float32)
        img.set_shape((size, size, c))
        label = tf.one_hot(label, 2)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(paths), 4096), seed=seed,
                        reshuffle_each_iteration=True)
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, len(paths)


def class_counts(csv_path: str, split: str) -> dict:
    _, labels = _read_manifest(csv_path, split)
    return {"Closed": labels.count(0), "Open": labels.count(1), "n": len(labels)}

## 5. 학습 CONFIG (`train_eye_mrl.py`)

실험 스위치는 이 셀만 바꾸면 된다.

- `SPLIT_MODE` : `"subject"` | `"random"` → leakage 영향 비교 (필수 비교 실험)
- `GRAYSCALE`  : `True` | `False` → 도메인 갭 실험
- `DO_SHARPEN` : `True` | `False` → sharpen on/off 실험 (학습·추론 동일값 필수)
- `INPUT_SIZE` : `96` | `128` | `256` → 입력 크기/속도 실험

In [ ]:
SPLIT_MODE = "subject"   # "subject" | "random"
GRAYSCALE  = True
DO_SHARPEN = False
INPUT_SIZE = 128
BATCH_SIZE = 64
EPOCHS     = 20
SEED       = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

MANIFEST = os.path.join(MANIFEST_DIR, f"mrl_manifest_{SPLIT_MODE}.csv")
TAG = f"{SPLIT_MODE}_{'gray' if GRAYSCALE else 'rgb'}_" \
      f"{'sharp' if DO_SHARPEN else 'nosharp'}_{INPUT_SIZE}"
MODEL_PATH   = os.path.join(ARTIFACT_DIR, f"eye_mrl_{TAG}.keras")
METRICS_PATH = os.path.join(ARTIFACT_DIR, f"eye_mrl_{TAG}_metrics.json")

print("CONFIG:", TAG)
print("manifest:", MANIFEST)
print("MODEL_PATH:", MODEL_PATH)

## 6. 모델 정의

In [ ]:
def build_eye_model(size: int, ch: int) -> tf.keras.Model:
    """기존 구조 유지 + 입력 파라미터화 + 회전 축소(눈 crop에 과하지 않게)."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(size, size, ch)),
        tf.keras.layers.Rescaling(1.0 / 255),
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.12, fill_mode="reflect"),  # 0.4->0.12
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(2, activation="softmax"),  # class0=Closed
    ])

## 7. 평가 함수

test 예측 → 지표 계산. Closed(=0)를 positive로 별도 보고한다
(졸음 감지에서 '감긴 눈을 Open으로 놓치는' Closed false-negative가 가장 위험하기 때문).

In [ ]:
def evaluate(model, test_ds) -> dict:
    """test 예측 -> 지표 계산. Closed(=0)를 positive로 별도 보고."""
    from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                                 confusion_matrix, classification_report)
    y_true, y_prob = [], []
    for xb, yb in test_ds:
        p = model(xb, training=False).numpy()
        y_prob.append(p)
        y_true.append(yb.numpy())
    y_prob = np.concatenate(y_prob)
    y_true = np.argmax(np.concatenate(y_true), axis=1)
    y_pred = np.argmax(y_prob, axis=1)

    acc = float(accuracy_score(y_true, y_pred))
    p_m, r_m, f_m, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    p_c, r_c, f_c, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0], average=None, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist()  # [[CC,CO],[OC,OO]]

    print("\n===== TEST 평가 =====")
    print(classification_report(y_true, y_pred, labels=[0, 1],
                                target_names=["Closed", "Open"], digits=4,
                                zero_division=0))
    print("Confusion Matrix  (행=실제, 열=예측) [Closed, Open]:")
    print(f"  실제 Closed: {cm[0]}")
    print(f"  실제 Open  : {cm[1]}")
    print(f"\n[졸음 관점] Closed-Recall={r_c[0]:.4f}  "
          f"Closed-Precision={p_c[0]:.4f}  Closed-F1={f_c[0]:.4f}")
    print("  * Closed-Recall이 낮으면 '감긴 눈을 Open으로 놓침' = 가장 위험")

    return {
        "tag": TAG,
        "accuracy": acc,
        "macro": {"precision": float(p_m), "recall": float(r_m), "f1": float(f_m)},
        "closed": {"precision": float(p_c[0]), "recall": float(r_c[0]),
                   "f1": float(f_c[0])},
        "confusion_matrix": cm,
    }

## 8. 데이터셋 로드

In [ ]:
for s in ("train", "val", "test"):
    print(f"  {s}:", class_counts(MANIFEST, s))

ch = channels(GRAYSCALE)
train_ds, n_tr = make_dataset(MANIFEST, "train", INPUT_SIZE, GRAYSCALE,
                              DO_SHARPEN, BATCH_SIZE, seed=SEED, mrl_root=MRL_ROOT)
val_ds, _ = make_dataset(MANIFEST, "val", INPUT_SIZE, GRAYSCALE,
                         DO_SHARPEN, BATCH_SIZE, mrl_root=MRL_ROOT)
test_ds, _ = make_dataset(MANIFEST, "test", INPUT_SIZE, GRAYSCALE,
                          DO_SHARPEN, BATCH_SIZE, mrl_root=MRL_ROOT)

## 9. 모델 컴파일

In [ ]:
model = build_eye_model(INPUT_SIZE, ch)
model.compile(optimizer="adam", loss="categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

## 10. 학습

val 기준 `ModelCheckpoint`로 모델 선택 + `EarlyStopping`. GPU 런타임에서 실행할 것.

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(MODEL_PATH), monitor="val_accuracy",
                                       save_best_only=True, mode="max"),
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4,
                                     mode="max", restore_best_weights=True),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

## 11. 최종 평가 (test, subject 독립) 및 저장

best 모델(val 기준 저장분)로 test set 1회 평가 후 메트릭을 JSON으로 저장한다.

In [ ]:
import json

best = tf.keras.models.load_model(MODEL_PATH)
metrics = evaluate(best, test_ds)
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("\n저장:", MODEL_PATH)
print("저장:", METRICS_PATH)